<!--
SPDX-FileCopyrightText: Copyright (c) 2024-2025 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

> **Note:** This file has been modified from the original
> [NVIDIA AI Blueprint: Bring Your LLM to NIM](https://github.com/NVIDIA-AI-Blueprints/bring-llms-to-nim).
> Changes: Adapted for Red Hat AI Inference Server (RHAIIS) on OpenShift,
> replacing NVIDIA NIM Docker deployment with OpenShift Kubernetes resources.

# Deploy GGUF Checkpoints with RHAIIS

This notebook shows you how to deploy memory-efficient quantized models using GGUF format with RHAIIS. Perfect for running large models on consumer GPUs or maximizing the number of models per server.

## What You'll Build

By the end of this notebook, you'll be able to:
- Deploy quantized models that use 50-75% less memory
- Run large models on consumer GPUs (8GB-16GB VRAM)
- Choose the right quantization level for your needs
- Handle GGUF's special configuration requirements

## When to Use This Approach

**Choose this notebook if you:**
- Have limited GPU memory (8GB-16GB VRAM)
- Want to run larger models on smaller GPUs
- Need to deploy multiple models on one GPU
- Can accept slight quality trade-offs for efficiency

## Understanding GGUF Quantization

Quantization reduces model size by using fewer bits for weights:

| Format | Model Size | Quality | Use Case |
|--------|------------|---------|----------|
| Full Precision | 100% (baseline) | Perfect | Research, fine-tuning |
| Q8_0 | ~33% | Near-perfect | Quality-focused deployment |
| Q5_K_M | ~22% | Excellent | Balanced deployment |
| Q4_K_M | ~18% | Very Good | **Recommended** - best balance |
| Q3_K_M | ~14% | Good | Memory-constrained |

**Example**: Llama-3.2-3B
- Full model: ~13GB -> Won't fit on RTX 3060
- Q4_K_M: ~2.1GB -> Runs comfortably on 8GB GPUs

## The GGUF Challenge

GGUF files don't include configuration metadata, so we need to:
1. Download the GGUF model file
2. Get the config.json from the original model
3. Organize them correctly for VLLM

Don't worry - we'll walk through this step-by-step!

## What's Covered

This tutorial includes:
* **Setup**: Understanding GGUF requirements
* **Example 1**: Deploying pre-downloaded GGUF models locally

## Setup: Python SDK with uv for IntelliJ

### Prerequisites Setup

This notebook uses `uv` for Python dependency management. Follow these steps to set up your environment:

#### 1. Initialize uv project
```bash
uv init --no-readme
```

#### 2. Add Jupyter dependencies to pyproject.toml
```bash
uv add jupyter notebook ipykernel ipywidgets
```

Or manually edit `pyproject.toml`:
```toml
dependencies = [
    "jupyter>=1.0.0",
    "notebook>=7.0.0",
    "ipykernel>=6.0.0",
    "ipywidgets>=8.0.0",
]
```

#### 3. Sync/install dependencies
```bash
uv sync
```

This creates a `.venv` virtual environment and installs all packages.

#### 4. Configure IntelliJ IDEA
1. Open **File → Project Structure → Project**
2. Click **SDK** → **Add SDK** → **Python SDK**
3. Select **Virtualenv Environment** → **Existing environment**
4. Browse to: `rhaiis-poc/.venv/bin/python`
5. Click **OK**

#### 5. Use the SDK in this notebook
1. In IntelliJ, select the Python interpreter (the one you just configured)
2. The notebook will use the Jupyter kernel from your `.venv`

---


## Setup OCP


#### 1. Create the Secret custom resource (CR) for the Hugging Face token. The cluster uses the Secret CR to pull models from Hugging Face.

1.1 Set the HF_TOKEN variable using the token you set in Hugging Face.

In [ ]:
!HF_TOKEN=<your_huggingface_token>

1.2 Set the cluster namespace to match where you deployed the Red Hat AI Inference Server image, for example:

In [ ]:
!NAMESPACE=rhaiis-namespace

1.3 Create the Secret CR in the cluster:

In [ ]:
!oc create secret generic hf-secret --from-literal=HF_TOKEN=$HF_TOKEN -n $NAMESPACE

1.4 Verify the Secret CR is ready:

In [ ]:
!oc get secret hf-secret -n $NAMESPACE

#### 2. Create the Docker secret so that the cluster can download the Red Hat AI Inference Server image from the container registry. For example, to create a Secret CR that contains the contents of your local ~/.docker/config.json file, run the following command:

In [ ]:
!oc create secret generic docker-secret --from-file=.dockercfg=$HOME/.docker/config.json --type=kubernetes.io/dockercfg -n $NAMESPACE

Verify the Docker secret is ready:

In [ ]:
!oc get secret docker-secret -n $NAMESPACE

## Utility Functions

Below are some utility functions we'll use in this notebook. These are for simplifying the process of deploying and monitoring in a notebook environment, and aren't required in general.

In [ ]:
import requests
import time


def check_service_ready(url):
  """Fallback health check using HTTP endpoint"""
  url = f"http://{url}/health"
  print("Checking service health endpoint...")

  while True:
    try:
      response = requests.get(url, headers={'accept': 'application/json'})
      if response.status_code == 200:
        print("✓ Service ready!")
        break
    except requests.ConnectionError:
      pass
    print("⏳ Still starting...")
    time.sleep(30)


def generate_text(url, model, prompt, max_tokens=1000, temperature=0.7):
  """Generate text using the service"""
  try:
    response = requests.post(
      f"http://{url}/v1/chat/completions",
      json={
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": temperature
      },
      timeout=60
    )
    response.raise_for_status()
    return response.json()['choices'][0]['message']['content']
  except requests.exceptions.RequestException as e:
    print(f"Error making request: {e}")
    return None


print("✓ Utility functions loaded successfully")

## GGUF Deployment Examples

Let's explore how to deploy GGUF models locally using RHAIIS.

## Example 1: Pre-download and Local GGUF Deployment

This example shows how to pre-download GGUF models and deploy them locally. This approach provides reliable offline usage and faster startup times since models are already available locally.

#### 3. Create a PersistentVolumeClaim (PVC) custom resource (CR) and apply it in the cluster. You use the PVC as the location where you store the models that you download.

In [ ]:
!oc apply -f ./1_download_local/pvc.yaml -n $NAMESPACE

#### 4. Download External Config File and GGUF Model Locally

In [ ]:
!oc apply -f ./1_download_local/download-job.yaml -n $NAMESPACE

#### 5. Create a Deployment custom resource (CR) that uses the GGUF model from the PVC and deploys the Red Hat AI Inference Server container.

In [ ]:
!oc apply -f ./3_local_model/deployment.yaml -n $NAMESPACE

#### 4. Create a Service CR for the model inference. For example:

In [ ]:
!oc apply -f ./3_local_model/service.yaml -n $NAMESPACE

#### 5. Create a Route CR to enable public access to the model. For example:

In [ ]:
!oc apply -f ./3_local_model/route.yaml

#### 6. Test Local GGUF Deployment:

In [ ]:
endpoint = !oc get route llama-3-2-3b-instruct-q8 -n $NAMESPACE -o jsonpath='{.spec.host}'

# URL is a list, access the first element
print(endpoint[0])

check_service_ready(url=endpoint[0])




Test the local model deployment:

In [ ]:
# Test Q8_0 quantization
result = generate_text(
  url=endpoint[0],
  model="meta-llama/Llama-3.2-3B-Instruct-Q8",
  prompt="Write a brief story about a robot learning to paint",
)
print("Q8_0 Quantization Result:")
print("=" * 50)
print(result if result else "Failed to generate text")

## Available Quantization Levels

The bartowski/Llama-3.2-3B-Instruct-GGUF repository includes multiple quantization levels. Each quantization requires its own directory with the GGUF file and configuration files.

### Directory Structure for Each Quantization:

Each quantization needs to be organized as follows:
```
quantization_directory/
├── config.json                    # From original model repo
├── tokenizer.json                 # From original model repo
├── tokenizer_config.json          # From original model repo
└── model_name-QUANTIZATION.gguf   # The quantized model file
```

## Summary

This notebook demonstrated deploying GGUF checkpoints with RHAIIS for memory-efficient model deployment.

**Key Accomplishments:**
- Set up complete GGUF deployment environment with NGC API keys and Docker
- Downloaded and deployed both Q4_K_M (~2.1GB) and Q8_0 (~3.2GB) quantizations of Llama-3.2-3B
- Handled GGUF's special requirements for external config files from original model repositories
- Achieved 50-75% memory reduction compared to full-precision models

**Technical Highlights:**
- GGUF models need external `config.json` files and separate directories per quantization
- Universal RHAIIS container supports GGUF format with proper configuration
- Q4_K_M offers best balance of quality/efficiency; Q8_0 provides near-perfect quality

**You Can Now:**
- Deploy large models on consumer GPUs (8GB-16GB VRAM)
- Set up offline deployments with faster startup times
- Choose quantization levels based on your quality/memory trade-offs

**Next Steps:** Explore other quantization levels, try different GGUF models from community repositories, and set up production deployments with monitoring.

For more GGUF models and documentation, check Hugging Face community repositories.
